In [1]:
import yaml
import matplotlib.pyplot as plt

def plot_metrics_to_pdf(yaml_path="training_metrics.yml"):
    # 1. Carrega os dados do YAML
    try:
        with open(yaml_path, "r") as f:
            history = yaml.safe_load(f)
    except FileNotFoundError:
        print(f"Erro: O arquivo {yaml_path} não foi encontrado.")
        return

    # 2. Prepara e extrai os dados agrupados por modo
    dados_por_modo = {}
    
    # Agora history é um dicionário onde a chave é o nome do modo
    for modo, metricas in history.items():
        dados_por_modo[modo] = {"times": [], "accs": [], "losses": []}
        
        # 'metricas' é a lista de épocas debaixo daquele modo
        for h in metricas:
            dados_por_modo[modo]["times"].append(h.get("train_time_cum"))
            
            # Padroniza a acurácia para a escala 0-100
            acc = h.get("accuracy")
            if acc is not None:
                acc = acc * 100 if acc <= 1.0 else acc
                dados_por_modo[modo]["accs"].append(acc)
                
            dados_por_modo[modo]["losses"].append(h.get("loss"))

    # 3. Configuração de Estética Global (Estilo Artigo Científico)
    plt.rcParams.update({
        'font.size': 12,
        'axes.grid': True,
        'grid.linestyle': '--',
        'grid.alpha': 0.6,
        'grid.color': '#b0b0b0',
        'lines.linewidth': 1.5,
        'lines.markersize': 5,
        'legend.framealpha': 1.0,
        'legend.edgecolor': 'black'
    })

    # ==========================================================
    # Gráfico 1: Cumulative Time x Accuracy
    # ==========================================================
    plt.figure(figsize=(8, 5)) # Proporção similar à da imagem
    
    for modo, dados in dados_por_modo.items():
        if dados["times"] and dados["accs"]:
            # LÓGICA DE ESTILO DA LINHA:
            # Se começar com "NS" usa linha contínua, senão usa pontilhada
            estilo_linha = '-' if modo.startswith("NS") else ':'
            
            plt.plot(dados["times"], dados["accs"], marker='o', linestyle=estilo_linha, label=modo)

    plt.xlabel("Cumulative Time (s)")
    plt.ylabel("Avg Test Accuracy")
    
    # Adiciona a legenda no canto inferior direito, como na imagem
    plt.legend(loc="lower right")

    plt.ylim(85, 95)
    
    # Salva o primeiro PDF
    plt.savefig("Time_x_Acc.pdf", format="pdf", bbox_inches="tight")
    plt.close() # Limpa a figura da memória

    # ==========================================================
    # Gráfico 2: Cumulative Time x Loss
    # ==========================================================
    plt.figure(figsize=(8, 5))
    
    for modo, dados in dados_por_modo.items():
        if dados["times"] and dados["losses"]:
            # A mesma lógica aplicada ao gráfico de Loss
            estilo_linha = '-' if modo.startswith("NS") else ':'
            
            plt.plot(dados["times"], dados["losses"], marker='o', linestyle=estilo_linha, label=modo)

    plt.xlabel("Cumulative Time (s)")
    plt.ylabel("Train Loss")
    
    # Loss geralmente cai, então a legenda fica melhor no canto superior direito
    plt.legend(loc="upper right")
    
    # Salva o segundo PDF
    plt.savefig("Time_x_Loss.pdf", format="pdf", bbox_inches="tight")
    plt.close()

    print("✅ Gráficos gerados com sucesso:")
    print("  - Time_x_Acc1.pdf")
    print("  - Time_x_Loss1.pdf")

if __name__ == "__main__":
    plot_metrics_to_pdf()

✅ Gráficos gerados com sucesso:
  - Time_x_Acc1.pdf
  - Time_x_Loss1.pdf
